# PokePlot, Lesson 1: Ask the Pokédex

**Can Python collect pictures of your favorite Pokémon?** Work alongside the lesson page, one question at a time: predict, try, and explain. By the end, collect three Pokémon images in a local folder.

Today's path: **URL → HTTP response → JSON → Python dictionaries/lists → image URL → image bytes → saved picture**. In future lessons, we will use the records to build NumPy arrays and plots.

Use Python 3.10+. If needed, run `%pip install requests pillow` in a setup cell. The installation name is `pillow`; its Python import is `PIL`.

**Fetch once, explore locally.** Save each successful Pokémon response and reuse the dictionary or file. Repeatedly inspecting a field does not require another network request. The same applies to downloaded images.

**Offline practice:** put the supplied `pikachu.json` beside this notebook. Skip the live request, save, and response-preview cells; use the local-file alternative below. The tiny JSON example still works offline. Downloading a new image requires internet unless you already saved that image.

Your working directory is where relative paths are resolved. Inspect it with `Path.cwd()` after importing `Path`.


## Q1. What does the API offer?

An **API**, or *application programming interface*, gives programs an agreed way to request information or actions. [PokéAPI](https://pokeapi.co/docs/v2) offers Pokémon data without an account or API key.

Open the [Pokémon endpoint documentation](https://pokeapi.co/docs/v2#pokemon). Browse the field descriptions instead of reading the whole page.

**Find:** Which field sounds useful if our goal is to get a Pokémon picture? Record its name and what you think its value will contain. Do you expect the actual picture or directions to find it?


My field: …

My prediction about its value: …


## Q2. Can you change the question in the URL?

Open <https://pokeapi.co/api/v2/pokemon/pikachu/> in a browser.

A **URL** is an address. An **endpoint** is an address where an API makes a resource or operation available. Here, `pokemon` selects the kind of resource, and `pikachu` selects a particular Pokémon.

**Predict:** What will change if you replace `pikachu` with `eevee`? Write your prediction, then try it and find the returned name.


Prediction: …

Observation: …


## Q3. Did the server answer successfully?

A **client** sends a request; a **server** sends a response. Your browser was the client. Now Python will take that role. **HTTP** is their exchange protocol; **GET** retrieves a resource.

Run this network cell once. `timeout=10` limits network waiting; it is not a guaranteed total download deadline.

**Explain:** What does the printed status code tell you?

| Code | Meaning |
| --- | --- |
| `200` | The request succeeded. |
| `404` | The resource was not found. |
| `500` | The server encountered an internal error. |


In [ ]:
import requests

url: str = "https://pokeapi.co/api/v2/pokemon/pikachu/"
response: requests.Response = requests.get(url, timeout=10)
print(response.status_code)


`response` is an HTTP response object, with a status, headers, and a body. A successful response does not mean Python has decoded its body yet.

Check the status before saving. `raise_for_status()` raises an exception for an HTTP error. `write_text` saves the body's text, and `_` receives the character count that we do not need. Skip these two response cells when using the offline snapshot.


In [ ]:
from pathlib import Path

response.raise_for_status()
cache_file: Path = Path("pikachu.json")
_ = cache_file.write_text(response.text, encoding="utf-8")


In [ ]:
raw_json: str = response.text
print(type(raw_json))
print(raw_json[:200])


The first 200 characters are just a preview; they can stop halfway through a value. Keep the complete text for decoding. The saved body does not contain the original response's status or headers.

If the network cannot connect, `requests.get` may raise an exception without creating a response. That differs from an HTTP error response such as `404`.


## Q4. Is JSON text already a dictionary?

**JSON**, or *JavaScript Object Notation*, is a text format for structured data. Python and many other languages can decode it. This tiny example keeps a few real Pikachu fields; the full record contains many more.

The triple quotes create **one Python string**. Its first character is `{`.

**Predict before running the last line:** What will `tiny_json[0]` return? Explain what the square brackets are indexing here.


In [ ]:
tiny_json: str = """{
  "name": "pikachu",
  "height": 4,
  "is_default": true,
  "types": [{"type": {"name": "electric"}}],
  "sprites": {
    "front_default": "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/25.png"
  }
}"""

print(type(tiny_json))
print(tiny_json[0])


## Q5. What changes when we decode it?

`json.loads` reads JSON **text** and creates Python objects. Here, `loads` means "load from a string." Use a JSON decoder rather than `eval`.

The annotation `dict[str, Any]` means a dictionary with string keys and values of different possible types. Annotations describe our expectations; they do not validate API data.

**Predict:** After decoding, what will `type(tiny_data["height"])` print? Run the cell, then compare a character position in `tiny_json` with a key lookup in `tiny_data`.


In [ ]:
import json
from typing import Any

tiny_data: dict[str, Any] = json.loads(tiny_json)
print(type(tiny_data))
print(type(tiny_data["height"]))


| JSON text describes | Python value after decoding |
| --- | --- |
| Object: `{...}` | `dict` |
| Array: `[...]` | `list` |
| String: `"pikachu"` | `str` |
| Whole number: `4` | `int` |
| `true`, `false`, `null` | `True`, `False`, `None` |

A URL inside JSON becomes a string. Decoding it does not fetch the image. Some picture fields are `null`, which becomes `None`: that means there is no image URL in that field.

Now decode the **full** response. This cell uses the existing response and makes no second request. Skip it if you are using the local-file alternative.

An object groups named values; an array keeps a sequence in order. In our example, `tiny_data["types"]` is a list, and `tiny_data["types"][0]` is its first dictionary. Use string keys for objects and integer positions for array entries.


In [ ]:
pokemon: dict[str, Any] = response.json()
print(type(pokemon))


### Local-file alternative

When offline or after a restart, skip the live-request and response cells. Uncomment and run this cell instead. It reads the saved JSON text, then decodes it; it does not recreate a live response, status, or headers. Leave it commented when following the live path.


In [ ]:
# import json
# from pathlib import Path
# from typing import Any
#
# cache_file: Path = Path("pikachu.json")
# raw_json: str = cache_file.read_text(encoding="utf-8")
# pokemon: dict[str, Any] = json.loads(raw_json)
# print(type(pokemon))


## Q6. How can you retrieve the name?

Inspect the top-level keys, then try the identifier example. A dictionary lookup uses a key; a list or string lookup can use a numerical position. The object being indexed determines the operation.

**Write:** Retrieve and print the Pokémon's name without printing the whole record. Annotate your variable.


In [ ]:
print(pokemon.keys())

pokemon_id: int = pokemon["id"]
print(pokemon_id)

# TODO: Retrieve and print the name using a str annotation.


## Q7. Where is the picture?

Return to the tiny JSON and find `sprites`. After decoding, that field contains another dictionary. The full record has more picture fields, but uses the same nested structure.

**Write:** Retrieve the URL under `front_default` from the full `pokemon` dictionary. Store it in `image_url: str | None` and print it. Trace the type after each lookup. The annotation means a URL string or `None`; a missing sprite has no URL to request.

Open the resulting URL in a browser to check your prediction. Looking up the URL in Python and opening that URL are separate actions.


In [ ]:
# TODO: Retrieve the nested image URL into image_url: str | None.
# TODO: Print it and check it in your browser.


## Q8. How do we save the picture itself?

The Pokémon record gave us the image's address. A **second GET request**, to that address, retrieves the picture. That response has a body of image **bytes**, rather than JSON describing a record.

`response.text` gives decoded text. `response.content` gives the body's bytes. For a PNG picture, use the bytes without trying to decode them as JSON.

After completing Q7, uncomment this scaffold. `mkdir` creates the image folder if needed. `len(image_bytes)` counts bytes, rather than pixels.

**Choose:** Should a `Path` save these bytes with `write_text` or `write_bytes`? Complete the final line and save the image.


In [ ]:
# if image_url is None:
#     raise ValueError("No front-facing image is available for this Pokémon.")
#
# image_response: requests.Response = requests.get(image_url, timeout=10)
# image_response.raise_for_status()
# print(image_response.headers.get("Content-Type"))
# image_bytes: bytes = image_response.content
# print(type(image_bytes))
# print(len(image_bytes))
#
# image_path: Path = Path("images") / f"{pokemon['name']}.png"
# image_path.parent.mkdir(parents=True, exist_ok=True)
# TODO: Write image_bytes to image_path using the appropriate Path method.


## Q9. How big is a picture?

After saving the image in Q8, uncomment this cell. Pillow reads the local image file; this does not send another request. Jupyter's `display` shows the decoded image.

`image.size` is `(width, height)` in **pixels**.

**Explain:** Is `len(image_bytes)` the same as the number of pixels, `width * height`? Predict, then calculate the pixel count from `image.size` and compare. Explain why these quantities describe different things.


In [ ]:
# from PIL import Image
# from IPython.display import display
#
# with Image.open(image_path) as image:
#     print(image.format)
#     print(image.size)
#     display(image)
#     # TODO: Calculate width * height and compare with len(image_bytes).


My explanation: …

The two requests we made, and what each response body contained: …


## Q10. Can you collect three favorites?

Choose three Pokémon, including the one you already fetched. Use the same steps to save their JSON records and their `front_default` pictures. Reuse cached files. A fresh name needs a fresh record, and each previously undownloaded image needs its own request.

**Make:** Display your three saved images with their Pokémon names. Check that each file exists in your `images` folder. This is a small gallery; no NumPy or Matplotlib is needed yet.

You may repeat the steps manually today. The function headers in [pokeplot-starter.py](/downloads/pokeplot/pokeplot-starter.py) will help turn that repeated work into reusable functions for homework.


In [ ]:
favorites: list[str] = ["pikachu", "eevee", "bulbasaur"]

# TODO: Replace these with three favorites.
# TODO: Fetch and save missing records, then their missing images.
# TODO: Display the saved images with their names.


## Homework: assemble your team gallery

Download [pokeplot-starter.py](/downloads/pokeplot/pokeplot-starter.py) beside this notebook, then fill in its function bodies one at a time. Use type annotations and test one function before moving on.

`json.dumps(pokemon)` converts a decoded record back into JSON text for saving with `write_text`.

A **type alias** gives a reusable name to a type. For example, `PokemonData = dict[str, Any]` lets a function header use `PokemonData` in place of the longer dictionary annotation. It still describes an expectation rather than checking the returned API data.

1. **One complete round trip:** finish the functions needed to fetch or reopen one Pokémon record, select its image URL, and save the image. Reuse the file on the second run.
2. **Your six-Pokémon team:** choose six favorites and use those functions to save their records and pictures. Make a local gallery labeled with their names. If part of your team is already cached, use those files.
3. **Check your work:** restart Python, load your saved records, and open the saved images without downloading them again. Explain which steps originally needed internet and which now run locally.
4. **A question worth exploring:** use the images to introduce your team, then propose one question about their numerical data. Name the fields you would need, make a prediction, and say what evidence could change your mind.

**Stretch:** collect normal and shiny pictures for the same Pokémon and display the pairs. Some image fields contain `None`. Decide what your function should do when no URL is available, then test that behavior with a small hand-made record before trying more API data.

## Sources

- [PokéAPI documentation](https://pokeapi.co/docs/v2)
- [Pokémon fields and sprite data](https://pokeapi.co/docs/v2#pokemon)
- [PokéAPI fair-use and caching policy](https://pokeapi.co/docs/v2#fairuse)
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Pillow Image reference](https://pillow.readthedocs.io/en/stable/reference/Image.html)

The companion snapshot is the complete Pikachu response body retrieved on October 18, 2026. It contains image URLs, not the image files themselves.
